# Notebook 07 — Learned selector controls

Use a **CPU runtime** and run the single code cell. Retain the original `SensorRepair_ICDSIS` Drive project and all Notebook 02, 04, 05 and 06 outputs. This notebook fits small tree models using cached neural outputs. No new neural-network training, neural test inference, dataset download, or manual data movement is needed.

The original benefit-and-harm policy, its thresholds and its predictions stay fixed. The new controls use the identical observable features, candidate keep/mask/reconstruct outputs, policy/validation/test participants, utility grid, and empirical validation constraints:

- **Ternary outcome selector:** one histogram gradient-boosting classifier predicts neutral, helpful and harmful outcomes jointly. Select a repair only if its predicted `P(help) − λ P(harm)` exceeds the validation-selected threshold.
- **Utility regression selector:** for each `λ ∈ {1,2,4}`, fit one squared-error gradient-boosting regressor to the action outcome `helpful − λ × harmful`. Predictions are clipped to the valid expected-utility range `[-λ,1]`. Select λ and threshold on validation with the same guards and tie-breaking as the original policy.

Targets are fitted on the **policy partition only**, using clean/dropout/bias/gain caches. No stuck-fault condition is used for fitting or threshold selection. Control settings are saved before test scoring. Internal random-row early stopping is disabled. Four control models are fitted per configuration, across all 15 configurations. Complete model fits and configurations are reused if the session pauses or disconnects. A 90-minute session budget pauses between fits/configurations; rerun the same cell to continue.

**Fairness and scope:** this is a comparison using the same features and data, not exact model-capacity matching. A multiclass tree ensemble can have more trees than a binary ensemble; the regression control fits three models. Training times, serialized model sizes, iterations and trees per iteration are recorded. These are study-specific controlled baselines, not claimed reproductions of a published SOTA method. The original outputs and all prior baselines/ablations are retained and reproduced.

**Already viewed test:** Notebook 06 results are known. This is explicitly an exploratory additional-control comparison, not a new untouched confirmation. The code never uses test labels to fit or select the controls, and never changes the original method. If a control is better, retain that outcome; do not change thresholds to recover a desired result. A revised proposed method would need fresh confirmatory evidence.

When all 15 configurations are complete, primary metrics use the original fold/condition/seed aggregation. Comparisons also include the separately labelled participant-balanced secondary metric and the conditional paired participant bootstrap from Notebook 06. Those intervals do not apply to the primary metric, do not resolve dependence from overlapping CV training data, and do not support population, significance, or certified-risk claims.

Related work already exists on [selective imputation](https://doi.org/10.1109/TKDE.2023.3240858), so novelty must be evaluated for the specific decision rule and experimental contribution. Implementation references: [HGB classifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html) and [HGB regressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html).

Run until **NOTEBOOK 07 COMPLETE** and **15/15**, then share the final printed output and the three named files. A partial seed is excluded from aggregate results. Earlier notebooks and outputs remain untouched.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc
import hashlib
import json
import os
import shutil
import time
import zlib
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from threadpoolctl import threadpool_limits

PROJECT=Path('/content/drive/MyDrive/SensorRepair_ICDSIS')
OUTPUT=PROJECT/'outputs'/'notebook07_v1'
LOCAL=Path('/content/sensorrepair_notebook07_v1')
CODE_VERSION='sensorrepair_nb07_v1'
CELL_SIGNATURE='de218dc067c4979a96f3c0531d6c943379fb18bf3e5dbd0f2e9c49035e6fdac3'
SESSION_MINUTES=90
MAX_ITER=160
HARM_PENALTIES=[1.0,2.0,4.0]
THRESHOLDS=[0.0,.005,.01,.02,.05,.1,2.0]
CLEAN_F1_TOLERANCE=.005
MAX_HARM=.01
BOOTSTRAP_RESAMPLES=10000
BOOTSTRAP_SEED=20261021
CONFIDENCE_LEVEL=.95
CONTROL_METHODS=['ternary_outcome_selector','utility_regression_selector']
ORIGINAL_METHODS=['keep','mask_suspect','reconstruct_suspect','residual_gate_mask','residual_gate_reconstruct',
                  'confidence_select','oracle_diagnostic','harm_aware_policy','benefit_only_policy',
                  'confidence_margin_tuned','residual_mask_tuned','residual_reconstruct_tuned']
METHODS=ORIGINAL_METHODS+CONTROL_METHODS
CONDITIONS=['clean']+[f'{f}_{i}' for f in ['dropout','bias','gain','stuck'] for i in [1,2,3]]
SCOPES={'clean':['clean'],'known_faults':CONDITIONS[1:10],'unseen_stuck':CONDITIONS[10:]}
NUMERIC=['macro_f1','accuracy','intervention_rate','helpful_rate','harmful_rate']
OUTPUT.mkdir(parents=True,exist_ok=True)
LOCAL.mkdir(parents=True,exist_ok=True)
SESSION_DEADLINE=time.monotonic()+SESSION_MINUTES*60
print('NOTEBOOK 07 | learned selector controls | CPU | one executable cell',flush=True)
print('Existing test results are already known. This is an exploratory control comparison.',flush=True)
print('Two new selectors, same features/actions/splits/validation constraints. Original policy is unchanged.',flush=True)



def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def json_signature(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)


def save_csv(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    value.to_csv(temporary, index=False)
    os.replace(temporary, path)


def save_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    with temporary.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(temporary, path)


def atomic_joblib(path, value):
    temporary = LOCAL / (Path(path).name + '.tmp')
    joblib.dump(value, temporary, compress=3)
    partial = Path(path).with_name(Path(path).name + '.partial')
    shutil.copyfile(temporary, partial)
    os.replace(partial, path)


def probability_statistics(probability):
    sorted_p = np.sort(probability, axis=-1)
    confidence = sorted_p[..., -1]
    margin = confidence - sorted_p[..., -2]
    entropy = -(probability * np.log(np.clip(probability, 1e-7, 1.0))).sum(axis=-1)
    return np.stack([confidence, margin, entropy], axis=-1)


def policy_features(probabilities, residuals, statistics, residual_scale):
    # This function accepts observable arrays only. It cannot access labels, subjects,
    # source IDs, injected fault locations, or condition/severity metadata.
    D, N, A, K = probabilities.shape
    G, J, M = residuals.shape[-1], A - 1, D * N
    p = probabilities.reshape(M, A, K)
    r = residuals.reshape(M, G)
    s = statistics.reshape(M, G, 4)
    original, candidates = p[:, 0], p[:, 1:]
    candidate_groups = np.tile(np.arange(G), 2)
    original_statistics = probability_statistics(original)
    candidate_statistics = probability_statistics(candidates)
    scaled = r / residual_scale[None, :]

    def repeat(values):
        return np.broadcast_to(values[:, None, :], (M, J, values.shape[-1]))

    middle = (original[:, None, :] + candidates) / 2.0
    js = 0.5 * (original[:, None, :] * np.log(np.clip(original[:, None, :], 1e-7, 1.0) / np.clip(middle, 1e-7, 1.0))).sum(axis=-1)
    js += 0.5 * (candidates * np.log(np.clip(candidates, 1e-7, 1.0) / np.clip(middle, 1e-7, 1.0))).sum(axis=-1)
    names, pieces = [], []

    def add(values, labels):
        assert values.shape == (M, J, len(labels))
        pieces.append(values)
        names.extend(labels)

    add(repeat(original), [f'original_probability_{k}' for k in range(K)])
    add(candidates, [f'action_probability_{k}' for k in range(K)])
    add(candidates - original[:, None, :], [f'probability_difference_{k}' for k in range(K)])
    add(repeat(original_statistics), ['original_confidence', 'original_margin', 'original_entropy'])
    add(candidate_statistics, ['action_confidence', 'action_margin', 'action_entropy'])
    add(candidate_statistics - original_statistics[:, None, :], ['confidence_difference', 'margin_difference', 'entropy_difference'])
    add(js[..., None], ['prediction_js_divergence'])
    add((candidates.argmax(axis=-1) == original.argmax(axis=-1)[:, None])[..., None], ['prediction_agreement'])
    kinds = np.zeros((J, 2), dtype=np.float32)
    kinds[:G, 0], kinds[G:, 1] = 1.0, 1.0
    add(np.broadcast_to(kinds[None], (M, J, 2)), ['action_is_mask', 'action_is_reconstruction'])
    add(np.broadcast_to(np.eye(G, dtype=np.float32)[candidate_groups][None], (M, J, G)), [f'candidate_group_{g}' for g in range(G)])
    add(repeat(np.log1p(r)), [f'log_residual_group_{g}' for g in range(G)])
    add(np.log1p(r[:, candidate_groups])[..., None], ['candidate_log_residual'])
    add(np.log1p(scaled[:, candidate_groups])[..., None], ['candidate_log_scaled_residual'])
    add(repeat(np.log1p(scaled.max(axis=-1, keepdims=True))), ['maximum_log_scaled_residual'])
    add(s[:, candidate_groups, :], ['candidate_mean_absolute_signal', 'candidate_signal_std', 'candidate_temporal_difference', 'candidate_zero_fraction'])
    add(repeat(s.reshape(M, G * 4)), [f'observed_group_{g}_stat_{j}' for g in range(G) for j in range(4)])
    values = np.ascontiguousarray(np.concatenate(pieces, axis=-1).reshape(M * J, -1), dtype=np.float32)
    assert len(set(names)) == len(names) and np.isfinite(values).all()
    return values, names


def outcome_targets(pack):
    predictions = pack['probabilities'].argmax(axis=-1)
    correct = predictions == pack['y'][None, :, None]
    original = correct[:, :, 0:1]
    fix = (~original & correct[:, :, 1:]).reshape(-1).astype(np.int8)
    harm = (original & ~correct[:, :, 1:]).reshape(-1).astype(np.int8)
    assert not np.any(fix & harm)
    return fix, harm


def macro_f1(y, prediction, classes):
    matrix = np.bincount(y * classes + prediction, minlength=classes * classes).reshape(classes, classes)
    numerator = 2.0 * matrix.diagonal()
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    return float(np.divide(numerator, denominator, out=np.zeros(classes), where=denominator > 0).mean())


def evaluate_choices(pack, choices, method):
    p, y = pack['probabilities'], pack['y']
    D, N, A, K = p.shape
    assert choices.shape == (D, N) and choices.min() >= 0 and choices.max() < A
    predictions = p.argmax(axis=-1)
    original_correct = predictions[:, :, 0] == y[None, :]
    selected = np.take_along_axis(predictions, choices[:, :, None], axis=2).squeeze(2)
    rows = []
    for i, condition in enumerate(pack['conditions']):
        correct = selected[i] == y
        helpful = float((~original_correct[i] & correct).mean())
        harmful = float((original_correct[i] & ~correct).mean())
        accuracy = float(correct.mean())
        assert abs(accuracy - original_correct[i].mean() - helpful + harmful) < 1e-10
        rows.append({'condition': condition, 'method': method, 'windows': N,
                     'macro_f1': macro_f1(y, selected[i], K), 'accuracy': accuracy,
                     'intervention_rate': float((choices[i] != 0).mean()), 'helpful_rate': helpful,
                     'harmful_rate': harmful,
                     'harm_given_original_correct': float(np.sum(original_correct[i] & ~correct) / max(1, original_correct[i].sum()))})
    return rows


def aggregate_rows(rows):
    clean = next(row for row in rows if row['condition'] == 'clean')
    corrupted = [row for row in rows if row['condition'] != 'clean']
    answer = {key: float(np.mean([row[key] for row in corrupted])) for key in
              ['macro_f1', 'accuracy', 'intervention_rate', 'helpful_rate', 'harmful_rate']}
    answer.update(clean_macro_f1=clean['macro_f1'], clean_accuracy=clean['accuracy'],
                  clean_harmful_rate=clean['harmful_rate'], clean_intervention_rate=clean['intervention_rate'])
    return answer


def baseline_choices(pack, scale):
    p, residual = pack['probabilities'], pack['residuals']
    D, N, A, K = p.shape
    G = residual.shape[-1]
    ratio = residual / scale[None, None, :]
    suspect = ratio.argmax(axis=-1)
    gate = ratio.max(axis=-1) > 1.0
    original_correct = p.argmax(axis=-1) == pack['y'][None, :, None]
    return {
        'keep': np.zeros((D, N), dtype=np.int64),
        'mask_suspect': 1 + suspect,
        'reconstruct_suspect': 1 + G + suspect,
        'residual_gate_mask': np.where(gate, 1 + suspect, 0),
        'residual_gate_reconstruct': np.where(gate, 1 + G + suspect, 0),
        'confidence_select': p.max(axis=-1).argmax(axis=-1),
        'oracle_diagnostic': original_correct.argmax(axis=-1)
    }


def pack_from_caches(paths, expected_signature):
    matrices, residuals, statistics = [], [], []
    anchor, conditions = None, []
    for path in paths:
        with np.load(path, allow_pickle=False) as saved:
            assert saved['signature'].item() == expected_signature
            identity = {key: saved[key] for key in ['y', 'subjects', 'window_ids', 'source_indices', 'action_names']}
            if anchor is None:
                anchor = identity
            else:
                for key, value in identity.items():
                    assert np.array_equal(value, anchor[key]), f'Cache alignment error: {key}'
            matrices.append(saved['probabilities'])
            residuals.append(saved['reconstruction_residuals'])
            statistics.append(saved['observed_group_statistics'])
            conditions.append(str(saved['condition'].item()))
    pack = dict(anchor)
    pack.update(probabilities=np.stack(matrices), residuals=np.stack(residuals),
                statistics=np.stack(statistics), conditions=conditions)
    p, r, s = pack['probabilities'], pack['residuals'], pack['statistics']
    D, N, A, K = p.shape
    G = r.shape[-1]
    assert A == 1 + 2 * G and r.shape == (D, N, G) and s.shape == (D, N, G, 4)
    assert np.isfinite(p).all() and np.isfinite(r).all() and np.isfinite(s).all()
    assert np.all(p >= 0) and np.allclose(p.sum(axis=-1), 1.0, atol=1e-5)
    assert np.all(r >= 0) and len(np.unique(pack['window_ids'])) == N
    return pack


def scope_summary(rows):
    result = []
    frame = pd.DataFrame(rows)
    for method in frame.method.unique():
        for scope, conditions in SCOPES.items():
            part = frame[(frame.method == method) & frame.condition.isin(conditions)]
            assert len(part) == len(conditions)
            result.append({'method': method, 'scope': scope, 'conditions': len(conditions),
                'unique_test_windows': int(part.windows.iloc[0]), **{key: float(part[key].mean()) for key in
                ['macro_f1', 'accuracy', 'intervention_rate', 'helpful_rate', 'harmful_rate']}})
    return result


def per_subject_results(pack, choices):
    # HAR individual participants need not contain every activity. Restrict to
    # that participant's observed classes for descriptive participant scores.
    result = []
    all_predictions = pack['probabilities'].argmax(axis=-1)
    K = pack['probabilities'].shape[-1]
    for method, actions in choices.items():
        if method == 'oracle_diagnostic':
            continue
        predictions = np.take_along_axis(all_predictions, actions[..., None], axis=2).squeeze(2)
        for subject in np.unique(pack['subjects']):
            use = pack['subjects'] == subject
            y = pack['y'][use]
            present = np.unique(y)
            for scope, conditions in SCOPES.items():
                scores, accuracies, helpful, harmful, interventions = [], [], [], [], []
                for name in conditions:
                    d = pack['conditions'].index(name)
                    pred = predictions[d, use]
                    cm = np.bincount(y*K+pred, minlength=K*K).reshape(K,K)
                    denominator = cm.sum(0) + cm.sum(1)
                    f1s = np.divide(2.0*cm.diagonal(), denominator, out=np.zeros(K), where=denominator>0)
                    correct = pred == y
                    original_correct = all_predictions[d, use, 0] == y
                    scores.append(float(f1s[present].mean()))
                    accuracies.append(float(correct.mean()))
                    helpful.append(float((~original_correct & correct).mean()))
                    harmful.append(float((original_correct & ~correct).mean()))
                    interventions.append(float((actions[d, use] != 0).mean()))
                result.append({'subject': int(subject), 'method': method, 'scope': scope,
                    'windows': int(use.sum()), 'classes_present': len(present),
                    'macro_f1_present_classes': float(np.mean(scores)), 'accuracy': float(np.mean(accuracies)),
                    'helpful_rate': float(np.mean(helpful)), 'harmful_rate': float(np.mean(harmful)),
                    'intervention_rate': float(np.mean(interventions))})
    return result


def assert_table_parity(computed,original,keys,columns):
    assert not computed.duplicated(keys).any() and not original.duplicated(keys).any()
    a=computed.set_index(keys).sort_index()
    b=original.set_index(keys).sort_index()
    assert a.index.equals(b.index),'Table row identities changed.'
    for col in columns:
        assert np.allclose(a[col].to_numpy(dtype=float),b[col].to_numpy(dtype=float),atol=1e-12,rtol=0,equal_nan=True),f'Metric parity failed: {col}'


def paired_bootstrap(differences,indices):
    # Each entry is one original participant's difference, already averaged over
    # that participant's three seeds and all conditions of the requested scope.
    values=np.asarray(differences,dtype=np.float64)
    assert values.ndim==1 and indices.ndim==2 and indices.shape[1]==len(values)
    samples=values[indices].mean(axis=1)
    tail=(1-CONFIDENCE_LEVEL)/2
    low,high=np.quantile(samples,[tail,1-tail])
    return float(values.mean()),float(low),float(high),samples




class PauseStudy(Exception):
    pass


def check_budget():
    if time.monotonic()>=SESSION_DEADLINE:
        raise PauseStudy('Session budget reached. Rerun this same cell to reuse completed model fits and configurations.')


STAGES={f'nb{i:02d}':PROJECT/'outputs'/f'notebook{i:02d}_v1' for i in [2,4,5,6]}
inventories={}
for stage,root in STAGES.items():
    p=root/f'notebook{int(stage[-2:]):02d}_artifact_inventory.json'
    if not p.is_file():raise FileNotFoundError(f'Missing {p}. Retain all Notebook 02, 04, 05 and 06 outputs.')
    records=json.loads(p.read_text())
    inventories[stage]={r['filename']:r for r in records}
    assert len(records)==len(inventories[stage])


def verified_copy(source,relative,expected):
    target=LOCAL/'inputs'/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.is_file() and file_sha256(target)==expected:return target
    partial=target.with_name(target.name+'.partial')
    shutil.copyfile(source,partial)
    assert file_sha256(partial)==expected,f'Input hash mismatch: {relative}'
    os.replace(partial,target)
    return target


def root_file(stage,relative):
    return verified_copy(STAGES[stage]/relative,stage+'/'+relative,inventories[stage][relative]['sha256'])


def root_json(stage,relative):
    return json.loads(root_file(stage,relative).read_text())


report5=root_json('nb05','notebook05_run_report.json')
manifest5=root_json('nb05','notebook05_study_manifest.json')
report6=root_json('nb06','notebook06_run_report.json')
assert report5['study_complete'] and report5['configurations_complete']==15
assert report5['study_signature']==manifest5['signature']==report6['source_study_signature']
assert report6['configurations_reproduced']==15 and report6['claim_audit']['full_study_reproduced_from_saved_predictions']
RESULTS=report5['completed_results']
SEEDS=[2026,2027,2028]
protocol=json.loads(verified_copy(PROJECT/'processed'/'v1'/'protocol.json','prepared/protocol.json',
                                  manifest5['plan']['protocol_sha256']).read_text())
PARAMETERS={'learning_rate':.05,'max_iter':MAX_ITER,'max_leaf_nodes':15,'max_depth':4,'min_samples_leaf':40,
            'l2_regularization':5.,'max_bins':64,'categorical_features':None,'early_stopping':False}
assert manifest5['plan']['policy']['max_iter']==MAX_ITER
assert manifest5['plan']['policy']['harm_penalties']==HARM_PENALTIES
assert manifest5['plan']['policy']['utility_thresholds']==THRESHOLDS
assert manifest5['plan']['policy']['max_clean_harm']==manifest5['plan']['policy']['max_corrupted_harm']==MAX_HARM
assert manifest5['plan']['policy']['clean_f1_tolerance']==CLEAN_F1_TOLERANCE
assert manifest5['plan']['environment']['sklearn']==sklearn.__version__, (
    'Use the original study scikit-learn version: '+manifest5['plan']['environment']['sklearn'])


def identity(job):
    return f'{job["dataset"]}_{job["fold"]}_seed_{job["seed"]}'


def original_run_file(result,filename):
    record=next(r for r in result['artifacts'] if r['filename']==filename)
    relative='runs/'+identity(result)+'/'+filename
    return verified_copy(STAGES['nb05']/relative,'nb05/'+relative,record['sha256'])


# Snapshot expected input hashes before fitting controls. Data bytes are verified when copied.
CACHE_INPUTS={}
for result in RESULTS:
    name=identity(result)
    mapping={}
    if result['imported_pilot']:
        cache_signature=root_json('nb02',name+'/dataset_complete.json')['cache_signature']
        test_signature=root_json('nb04','notebook04_frozen_manifest.json')['signature']
        for split in ['policy','validation','test']:
            stage='nb04' if split=='test' else 'nb02'
            signature=test_signature if split=='test' else cache_signature
            for condition in (CONDITIONS if split=='test' else CONDITIONS[:10]):
                relative=name+f'/action_cache/{split}__{condition}.npz'
                entry=inventories[stage][relative]
                mapping[split+'|'+condition]={'stage':stage,'filename':relative,'sha256':entry['sha256'],'signature':signature}
    else:
        run=STAGES['nb05']/'runs'/name
        signature=json_signature({'run':result['signature'],'classifier':file_sha256(run/'classifier_best.pt'),
                                  'reconstructor':file_sha256(run/'reconstructor_best.pt')})
        for split in ['policy','validation','test']:
            for condition in (CONDITIONS if split=='test' else CONDITIONS[:10]):
                relative='runs/'+name+f'/action_cache/{split}__{condition}.npz'
                digest=json.loads((STAGES['nb05']/(relative+'.sha256.json')).read_text())
                assert digest['signature']==signature,'Cached neural-model fingerprint changed.'
                mapping[split+'|'+condition]={'stage':'nb05','filename':relative,'sha256':digest['sha256'],'signature':signature}
    CACHE_INPUTS[name]=mapping
plan={'code_version':CODE_VERSION,'cell_signature':CELL_SIGNATURE,'source_study_signature':report5['study_signature'],
      'source_audit_signature':report6['analysis_signature'],
      'source_report_sha256':inventories['nb05']['notebook05_run_report.json']['sha256'],
      'source_audit_sha256':inventories['nb06']['notebook06_run_report.json']['sha256'],
      'cache_inputs':CACHE_INPUTS,'jobs':[{k:r[k] for k in ['dataset','fold','seed']} for r in RESULTS],
      'model_parameters':PARAMETERS,'harm_penalties':HARM_PENALTIES,'thresholds':THRESHOLDS,
      'selection_guards':{'clean_f1_tolerance':CLEAN_F1_TOLERANCE,'max_clean_harm':MAX_HARM,'max_corrupted_harm':MAX_HARM},
      'training_partition':'policy only; ten development conditions, no stuck faults',
      'selection_partition':'validation only; identical original utility grid and empirical constraints',
      'new_control_definitions':{
          'ternary_outcome_selector':'one HGB classifier predicts neutral=0, helpful=1, harmful=2; score P(help)-lambda*P(harm)',
          'utility_regression_selector':'one HGB squared-error regressor per lambda predicts helpful-lambda*harmful; score clipped to [-lambda,1]'},
      'capacity_note':'same observable features and per-model limits; not exact capacity matching. Ternary boosting may use more trees; utility regression fits three models.',
      'original_method_and_predictions':'preserved without refitting or retuning',
      'evaluation_status':'exploratory comparison after viewing original test outcomes; not a new untouched test',
      'neural_training_or_inference_required':False,'classical_control_models_per_configuration':4,
      'aggregation':report5['aggregation'],
      'bootstrap':'same participant-balanced secondary metric and conditional paired percentile bootstrap as Notebook 06',
      'bootstrap_resamples':BOOTSTRAP_RESAMPLES,'bootstrap_seed':BOOTSTRAP_SEED,
      'software':{'sklearn':sklearn.__version__,'numpy':np.__version__},
      'references':[
          'https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html',
          'https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html',
          'https://doi.org/10.1109/TKDE.2023.3240858']}
STUDY_SIGNATURE=json_signature(plan)
manifest_path=OUTPUT/'notebook07_study_manifest.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text())['signature']==STUDY_SIGNATURE,'Control study inputs or settings changed. Restore the fixed configuration.'
else:save_json(manifest_path,{'signature':STUDY_SIGNATURE,'plan':plan})
print('Control comparison plan frozen:',STUDY_SIGNATURE[:16],flush=True)


def job_signature(job):
    return json_signature({'study':STUDY_SIGNATURE,**job})


def load_pack(result,split):
    assert split in ['policy','validation','test']
    paths=[];expected_signature=None
    for condition in (CONDITIONS if split=='test' else CONDITIONS[:10]):
        record=CACHE_INPUTS[identity(result)][split+'|'+condition]
        path=verified_copy(STAGES[record['stage']]/record['filename'],record['stage']+'/'+record['filename'],record['sha256'])
        if expected_signature is None:expected_signature=record['signature']
        assert expected_signature==record['signature']
        with np.load(path,allow_pickle=False) as saved:
            assert saved['split'].item()==split and saved['condition'].item()==condition
        paths.append(path)
    pack=pack_from_caches(paths,expected_signature)
    assert pack['conditions']==(CONDITIONS if split=='test' else CONDITIONS[:10])
    assert np.array_equal(np.unique(pack['y']),np.arange(pack['probabilities'].shape[-1]))
    return pack


def fit_control(name,X,target,kind,job,run,signature):
    path=run/(name+'.joblib')
    if path.is_file():
        payload=joblib.load(path)
        assert payload['signature']==signature and payload['kind']==kind
        print('  Reused control model:',name,flush=True)
        return payload['model'],{**payload['cost'],'saved_bytes':path.stat().st_size}
    check_budget();started=time.perf_counter()
    unique=np.unique(target)
    if len(unique)==1:
        model={'constant':float(unique[0]),'kind':kind}
    else:
        cls=HistGradientBoostingClassifier if kind=='ternary' else HistGradientBoostingRegressor
        model=cls(**PARAMETERS,random_state=int(job['seed']))
        model.fit(X,target)
    seconds=time.perf_counter()-started
    cost={**job,'model':name,'kind':kind,'training_rows':len(X),'features':X.shape[1],
          'target_values':unique.tolist(),'training_seconds':seconds,
          'iterations':int(getattr(model,'n_iter_',0)),
          'trees_per_iteration':int(getattr(model,'n_trees_per_iteration_',0))}
    atomic_joblib(path,{'signature':signature,'kind':kind,'model':model,'cost':cost})
    cost['saved_bytes']=path.stat().st_size
    print(f'  Fitted {name}: {len(X)} action-case rows | {seconds:.1f}s',flush=True)
    return model,cost


def ternary_probabilities(model,X):
    output=np.zeros((len(X),3),dtype=np.float32)
    if isinstance(model,dict):
        output[:,int(model['constant'])]=1.
    else:
        output[:,model.classes_.astype(int)]=model.predict_proba(X).astype(np.float32)
    assert np.allclose(output.sum(1),1,atol=1e-5)
    return output


def control_scores(pack,scale,feature_names,models):
    X,names=policy_features(pack['probabilities'],pack['residuals'],pack['statistics'],scale)
    assert names==feature_names
    shape=pack['probabilities'].shape[:2]+(pack['probabilities'].shape[2]-1,)
    p=ternary_probabilities(models['ternary'],X)
    scores={family:{} for family in CONTROL_METHODS}
    for penalty in HARM_PENALTIES:
        scores['ternary_outcome_selector'][penalty]=(p[:,1]-penalty*p[:,2]).reshape(shape)
        model=models['utility_'+str(penalty)]
        values=(np.full(len(X),model['constant']) if isinstance(model,dict) else model.predict(X)).astype(np.float32)
        scores['utility_regression_selector'][penalty]=np.clip(values,-penalty,1.).reshape(shape)
    assert all(np.isfinite(a).all() for group in scores.values() for a in group.values())
    return scores


def actions_from_scores(score,threshold):
    best=score.argmax(-1)
    return np.where(score.max(-1)>threshold,best+1,0)


def select_control(pack,scores,family):
    keep=np.zeros(pack['probabilities'].shape[:2],dtype=np.int64)
    clean_keep=aggregate_rows(evaluate_choices(pack,keep,'keep'))['clean_macro_f1']
    best=None;trials=[]
    for penalty in HARM_PENALTIES:
        for threshold in THRESHOLDS:
            actions=actions_from_scores(scores[family][penalty],threshold)
            metrics=aggregate_rows(evaluate_choices(pack,actions,family))
            feasible=(metrics['clean_macro_f1']>=clean_keep-CLEAN_F1_TOLERANCE-1e-12
                and metrics['clean_harmful_rate']<=MAX_HARM+1e-12 and metrics['harmful_rate']<=MAX_HARM+1e-12)
            settings={'family':family,'harm_penalty':penalty,'threshold':threshold}
            trials.append({'family':family,'harm_penalty':penalty,'threshold':threshold,'feasible':bool(feasible),**metrics})
            rank=(metrics['macro_f1'],metrics['clean_macro_f1'],-metrics['harmful_rate'],-metrics['intervention_rate'])
            if feasible and (best is None or rank>best['rank']):
                best={'rank':rank,'settings':settings,'summary':metrics,'keep_only_on_validation':bool(np.all(actions==0))}
    assert best is not None,'The explicit keep candidate must remain feasible.'
    return {k:v for k,v in best.items() if k!='rank'},trials


def evaluate_job(result):
    job={key:result[key] for key in ['dataset','fold','seed']}
    run=OUTPUT/'runs'/identity(job);run.mkdir(parents=True,exist_ok=True)
    signature=job_signature(job)
    selected_original=json.loads(original_run_file(result,'selected_policy.json').read_text())
    policy,validation=load_pack(result,'policy'),load_pack(result,'validation')
    assert not set(policy['subjects'].tolist())&set(validation['subjects'].tolist())
    assert not set(policy['window_ids'].tolist())&set(validation['window_ids'].tolist())
    scale=np.maximum(np.quantile(policy['residuals'][0],.95,axis=0),1e-8)
    assert np.allclose(scale,selected_original['residual_scale'],atol=1e-12,rtol=1e-12)
    X,feature_names=policy_features(policy['probabilities'],policy['residuals'],policy['statistics'],scale)
    assert feature_names==selected_original['feature_names']
    assert policy['action_names'].tolist()==validation['action_names'].tolist()==selected_original['action_names']
    fix,harm=outcome_targets(policy)
    ternary=np.where(fix==1,1,np.where(harm==1,2,0)).astype(np.int8)
    models={};costs=[]
    models['ternary'],cost=fit_control('ternary',X,ternary,'ternary',job,run,signature);costs.append(cost)
    for penalty in HARM_PENALTIES:
        name='utility_'+str(penalty)
        target=fix.astype(np.float32)-np.float32(penalty)*harm.astype(np.float32)
        models[name],cost=fit_control(name,X,target,'utility',job,run,signature);costs.append(cost)
    del X,fix,harm,ternary
    validation_scores=control_scores(validation,scale,feature_names,models)
    selections={};trial_rows=[]
    for family in CONTROL_METHODS:
        selections[family],trials=select_control(validation,validation_scores,family);trial_rows.extend(trials)
    saved_selection={'signature':signature,'selection_split':'validation','selected_controls':selections,
                     'feature_names':feature_names,'policy_participants':np.unique(policy['subjects']).tolist(),
                     'validation_participants':np.unique(validation['subjects']).tolist(),
                     'note':'Test outcomes were not used to fit or select these controls; the broader comparison is exploratory.'}
    selection_path=run/'selected_controls.json'
    if selection_path.exists():assert json.loads(selection_path.read_text())==saved_selection,'Saved control selection changed.'
    else:save_json(selection_path,saved_selection)
    save_csv(run/'validation_trials.csv',pd.DataFrame(trial_rows))
    del validation,validation_scores
    check_budget()
    # First access to test arrays occurs only AFTER control thresholds are committed.
    test=load_pack(result,'test')
    assert not set(test['subjects'].tolist())&(set(saved_selection['policy_participants'])|set(saved_selection['validation_participants']))
    assert test['action_names'].tolist()==selected_original['action_names']
    with np.load(original_run_file(result,'test_predictions.npz'),allow_pickle=False) as saved:
        assert saved['signature'].item()==result['signature']
        assert saved['methods'].tolist()==ORIGINAL_METHODS and saved['conditions'].tolist()==CONDITIONS
        for key in ['y','subjects','window_ids','source_indices']:assert np.array_equal(saved[key],test[key])
        choices={m:saved['choices'][i] for i,m in enumerate(ORIGINAL_METHODS)}
        old_predictions=saved['predictions']
    all_predictions=test['probabilities'].argmax(-1)
    for i,method in enumerate(ORIGINAL_METHODS):
        actual=np.take_along_axis(all_predictions,choices[method][...,None],axis=2).squeeze(2)
        assert np.array_equal(actual,old_predictions[i]),'Cached candidate predictions differ from the original study.'
    for method,actions in baseline_choices(test,scale).items():
        assert np.array_equal(actions,choices[method]),f'Original baseline action parity failed: {method}'
    original_rows=[{**job,**row} for method in ORIGINAL_METHODS for row in evaluate_choices(test,choices[method],method)]
    original_summary=[{**job,**row} for row in scope_summary(original_rows)]
    assert_table_parity(pd.DataFrame(original_summary),pd.read_csv(original_run_file(result,'test_summary.csv')),
                       ['dataset','fold','seed','method','scope'],NUMERIC)
    scores=control_scores(test,scale,feature_names,models)
    for family,item in selections.items():
        settings=item['settings']
        choices[family]=actions_from_scores(scores[family][settings['harm_penalty']],settings['threshold'])
        print(f'  {family}: {settings} | test intervention {np.mean(choices[family]!=0)*100:.2f}%',flush=True)
    rows=[{**job,**row} for method,actions in choices.items() for row in evaluate_choices(test,actions,method)]
    summary=[{**job,**row} for row in scope_summary(rows)]
    participants=[{**job,**row} for row in per_subject_results(test,choices)]
    predictions=np.stack([np.take_along_axis(all_predictions,choices[m][...,None],axis=2).squeeze(2) for m in METHODS])
    save_npz(run/'test_predictions.npz',signature=np.asarray(signature),methods=np.asarray(METHODS),conditions=np.asarray(CONDITIONS),
             predictions=predictions,choices=np.stack([choices[m] for m in METHODS]),
             **{key:test[key] for key in ['y','subjects','window_ids','source_indices']})
    for name,values in [('per_condition',rows),('test_summary',summary),('per_participant',participants),('model_costs',costs)]:
        save_csv(run/(name+'.csv'),pd.DataFrame(values))
    output={**job,'signature':signature,'original_candidate_prediction_parity':True,'original_baseline_action_parity':True,
            'original_summary_parity':True,'selected_controls':selections,'policy_windows':len(policy['y']),
            'test_windows':len(test['y']),'feature_count':len(feature_names)}
    output['artifacts']=[{'filename':name,'sha256':file_sha256(run/name)} for name in
        ['test_predictions.npz','selected_controls.json','per_condition.csv','test_summary.csv','per_participant.csv','model_costs.csv']]
    save_json(run/'dataset_complete.json',output)
    return output


def completed(result):
    path=OUTPUT/'runs'/identity(result)/'dataset_complete.json'
    if not path.is_file():return None
    data=json.loads(path.read_text())
    job={k:result[k] for k in ['dataset','fold','seed']}
    assert data['signature']==job_signature(job)
    for record in data['artifacts']:assert file_sha256(path.parent/record['filename'])==record['sha256'],'Completed result changed.'
    return data


def publish_progress(final=False):
    done=[data for result in RESULTS if (data:=completed(result)) is not None]
    if not done:
        if final:print('NOTEBOOK 07 PAUSED — no complete configuration yet; model fits are saved.',flush=True)
        return
    tables={}
    for name in ['test_summary','per_condition','per_participant','model_costs']:
        tables[name]=pd.concat([pd.read_csv(OUTPUT/'runs'/identity(r)/(name+'.csv')) for r in done],ignore_index=True)
        save_csv(OUTPUT/('notebook07_'+name+'.csv'),tables[name])
    summary=tables['test_summary'];numeric=NUMERIC
    seeds=[]
    for keys,part in summary.groupby(['dataset','seed','method','scope'],sort=False):
        dataset,seed,method,scope=keys
        required=4 if dataset=='dsads' else 1
        assert len(part)==part.fold.nunique()<=required
        seeds.append({'dataset':dataset,'seed':int(seed),'method':method,'scope':scope,'folds_complete':len(part),
                      'complete_for_seed':len(part)==required,**{c:float(part[c].mean()) for c in numeric}})
    seed_frame=pd.DataFrame(seeds);save_csv(OUTPUT/'notebook07_per_seed_summary.csv',seed_frame)
    aggregated=[]
    for keys,part in seed_frame[seed_frame.complete_for_seed].groupby(['dataset','method','scope'],sort=False):
        row=dict(zip(['dataset','method','scope'],keys));row['complete_seeds']=len(part)
        for c in numeric:
            row[c+'_mean']=float(part[c].mean());row[c+'_seed_sd']=float(part[c].std(ddof=1)) if len(part)>1 else None
        aggregated.append(row)
    aggregate=pd.DataFrame(aggregated)
    save_csv(OUTPUT/'notebook07_aggregate_summary.csv',aggregate)
    ready=len(done)==15;comparison=[]
    if ready:
        old=pd.read_csv(root_file('nb05','notebook05_aggregate_summary.csv'))
        assert_table_parity(aggregate[aggregate.method.isin(ORIGINAL_METHODS)],old,['dataset','method','scope'],
                           [c+suffix for c in numeric for suffix in ['_mean','_seed_sd']])
        lookup=aggregate.set_index(['dataset','method','scope'])
        subject=tables['per_participant'].groupby(['dataset','subject','method','scope'],sort=False).macro_f1_present_classes.mean().reset_index()
        for dataset in ['dsads','har']:
            people=np.sort(subject[subject.dataset==dataset].subject.unique())
            rng=np.random.default_rng(BOOTSTRAP_SEED+zlib.crc32(dataset.encode()))
            indices=rng.integers(0,len(people),size=(BOOTSTRAP_RESAMPLES,len(people)),dtype=np.int16)
            for scope in SCOPES:
                pivot=subject[(subject.dataset==dataset)&(subject.scope==scope)].pivot(index='subject',columns='method',values='macro_f1_present_classes').loc[people]
                run_scores=summary[(summary.dataset==dataset)&(summary.scope==scope)].pivot(index=['fold','seed'],columns='method',values='macro_f1')
                for control in CONTROL_METHODS:
                    a,b=lookup.loc[(dataset,'harm_aware_policy',scope)],lookup.loc[(dataset,control,scope)]
                    point,lo,hi,_=paired_bootstrap((pivot.harm_aware_policy-pivot[control]).to_numpy()*100,indices)
                    delta=100*(run_scores.harm_aware_policy-run_scores[control])
                    comparison.append({'dataset':dataset,'scope':scope,'control':control,
                        'original_policy_macro_f1_pct':float(a.macro_f1_mean*100),'control_macro_f1_pct':float(b.macro_f1_mean*100),
                        'primary_policy_minus_control_pp':float(100*(a.macro_f1_mean-b.macro_f1_mean)),
                        'original_policy_harm_pct':float(a.harmful_rate_mean*100),'control_harm_pct':float(b.harmful_rate_mean*100),
                        'positive_runs':int((delta>1e-10).sum()),'negative_runs':int((delta<-1e-10).sum()),
                        'tied_runs':int((delta.abs()<=1e-10).sum()),'participants':len(people),
                        'secondary_participant_balanced_delta_pp':point,'secondary_bootstrap_low_pp':lo,'secondary_bootstrap_high_pp':hi,
                        'evaluation_status':'exploratory; intervals conditional on saved fits and refer only to the secondary metric'})
        save_csv(OUTPUT/'notebook07_policy_vs_controls.csv',pd.DataFrame(comparison))
    output={'code_version':CODE_VERSION,'study_signature':STUDY_SIGNATURE,'study_complete':ready,
        'configurations_complete':len(done),'configurations_required':15,'completed_results':done,
        'pending_configurations':[{k:r[k] for k in ['dataset','fold','seed']} for r in RESULTS if identity(r) not in {identity(d) for d in done}],
        'new_neural_training_runs':0,'new_neural_test_inference':0,'original_policy_refit_or_retuned':False,
        'classical_control_models_per_configuration':4,'test_status':'previously inspected; exploratory additional controls',
        'capacity_note':plan['capacity_note'],'aggregation':report5['aggregation'],
        'secondary_interval_note':'participant-balanced effects only, conditional on saved models; no simultaneous or population guarantee',
        'p_values_or_significance_claim':False,'capacity_or_compute_savings_claim':False,'published_method_reimplementation_claim':False,
        'comparison_summary':comparison,'output_directory':str(OUTPUT)}
    save_json(OUTPUT/'notebook07_run_report.json',output)
    files=[{'filename':p.name,'bytes':p.stat().st_size,'sha256':file_sha256(p)} for p in sorted(OUTPUT.glob('notebook07_*'))
           if p.is_file() and not p.name.endswith('.partial') and p.name!='notebook07_artifact_inventory.json']
    save_json(OUTPUT/'notebook07_artifact_inventory.json',files)
    if not final:
        print(f'Progress saved: {len(done)}/15 configurations.',flush=True);return
    print('\nNOTEBOOK 07 '+('COMPLETE' if ready else 'PAUSED — RERUN THE SAME CELL'),flush=True)
    print(f'Configurations complete: {len(done)}/15 | Neural training/inference: 0 | Original thresholds changed: 0',flush=True)
    if ready:
        display=pd.DataFrame(comparison)
        display=display[display.scope!='clean']
        print('POLICY MINUS NEW CONTROLS | primary Macro-F1 percentage points; positive favors original policy',flush=True)
        print(display[['dataset','scope','control','original_policy_macro_f1_pct','control_macro_f1_pct',
                       'primary_policy_minus_control_pp','original_policy_harm_pct','control_harm_pct','positive_runs','negative_runs','tied_runs']]
              .to_string(index=False,float_format=lambda x:f'{x:.3f}'),flush=True)
        print('All original aggregate results reproduced. All new control settings were selected on validation only.',flush=True)
    print('This is an exploratory comparison after inspection of the original test results.',flush=True)
    print('Outputs:',OUTPUT,flush=True)
    print('When 15/15 complete, share the final output and notebook07_aggregate_summary.csv, notebook07_run_report.json, notebook07_policy_vs_controls.csv.',flush=True)


def main():
    try:
        for result in RESULTS:
            if completed(result) is not None:
                print('REUSE',identity(result),flush=True);continue
            check_budget();print('\nRUN',identity(result),flush=True)
            evaluate_job(result);gc.collect();publish_progress()
    except PauseStudy as exc:print(str(exc),flush=True)
    finally:publish_progress(final=True)


with threadpool_limits(limits=2):main()


Mounted at /content/drive
NOTEBOOK 07 | learned selector controls | CPU | one executable cell
Existing test results are already known. This is an exploratory control comparison.
Two new selectors, same features/actions/splits/validation constraints. Original policy is unchanged.
Control comparison plan frozen: 65f7f369c33a1c0d

RUN dsads_fold_0_seed_2026
  Fitted ternary: 114000 action-case rows | 29.5s
  Fitted utility_1.0: 114000 action-case rows | 7.4s
  Fitted utility_2.0: 114000 action-case rows | 8.4s
  Fitted utility_4.0: 114000 action-case rows | 7.7s
  ternary_outcome_selector: {'family': 'ternary_outcome_selector', 'harm_penalty': 4.0, 'threshold': 0.005} | test intervention 4.59%
  utility_regression_selector: {'family': 'utility_regression_selector', 'harm_penalty': 2.0, 'threshold': 0.02} | test intervention 12.55%
Progress saved: 1/15 configurations.

RUN dsads_fold_1_seed_2026
  Fitted ternary: 114000 action-case rows | 28.9s
  Fitted utility_1.0: 114000 action-case rows